In [1]:
from dataloader import *
from augmentations import *
from models import *
from training import *
import matplotlib.pyplot as plt
from itertools import product
from statistics import mean, stdev

## Models

In [2]:
models = ["ResNet",
           "ConvNeXt", 
           "VisionTransformer", 
           "SmallCNN"]

## Experiment loop for hyperparameter selection

In [13]:
params = {
     #training params
    "batch_size": [8, 32, 128],
    "learning_rate": [1e-3, 1e-4, 1e-5],
    "optimizer": ['Adam', 'SGD'],
    #regularization params
    "dropout": [0.2, 0.5],
    "weight_decay": [0, 1e-3, 0.5]
}

# Create experiment grid
experiment_grid = [dict(zip(params.keys(), v)) for v in product(*params.values())]

In [ ]:
def run_experiment(model_name,experiment_config, epoch_number=2):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # model
    model = get_model(model_name)   
    model.to(device)


    # optimizer
    if experiment_config["optimizer"] == "adam":
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"]
        )
    else:
        optimizer = torch.optim.SGD(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"],
            momentum=0.9
        )

    # data

    transformations = basic_transforms(augmentation_type=None, model_type=model_name)

    train_dataset = get_train_dataset(transform=transformations)
    val_dataset, test_dataset = get_val_train_dataset(model_type=model_name)

    
    train_loader = get_train_dataloaders(
        train_dataset,
        collate_fn=None,
        batch_size=experiment_config["batch_size"]
    )
    
    val_loader, test_loader = get_val_test_dataloaders(
        val_dataset, test_dataset,
        batch_size=experiment_config["batch_size"]
    )

    criterion = torch.nn.CrossEntropyLoss()

    accs = []
    f1s = []
    recalls = []
    precisions = []
    for epoch in range(epoch_number):

        loss_score = train(model, train_loader, optimizer, criterion, device)
        scores = validate(model, val_loader, device,criterion)

        print(f"Epoch number: {epoch}; training loss: {loss_score:.5f}; val loss: {scores['loss']:.5f}")

        accs.append(scores["accuracy"])
        f1s.append(scores["f1"])
        recalls.append(scores["recall"])
        precisions.append(scores["precision"])

    val_scores = {
        'mean_acc': mean(accs),
        'std_acc': stdev(accs),

        'mean_f1': mean(f1s),
        'std_f1': stdev(f1s),

        'mean_recall': mean(recalls),
        'std_recall': stdev(recalls),
        'mean_precision': mean(precisions),
        'std_precision': stdev(precisions)
    }
    test_scores = validate(model, test_loader, device, criterion)

    return {**val_scores, **test_scores}

In [ ]:
results = []

for model in models:

    for i, config in enumerate(experiment_grid):

        if i ==3:
            break

        print(f"Configuration number {i+1}, model: {model} config params: {config}")


        score = run_experiment(model,config)
        d = {
            "model": model,
            "config": config}
        results.append({
            **d, **score
        })

## Model Ranking

In [ ]:
for model in models:
    model_results = list(filter(lambda res: res["model"] == model, results))
    sorted_results = sorted(model_results, key=lambda x: -x["accuracy"])
    print(f"Ranking for model {model}")
    for i,res in enumerate(sorted_results):
        print("Rank number: ",i, end=" ")
        print(sorted_results[i])
    print("="*50)

### Tutaj wybieramy najlepsze konfiguracje (jedna na kazdy model) i zaczynamy testowanie augmentacji

## Experiment loop for auqmentation techniques

In [14]:
# przykladowo
best_conf_resnet = {'batch_size': 8, 'learning_rate': 0.001, 'optimizer': 'Adam', 'dropout': 0.2, 'weight_decay': 0.001}
best_conf_smallcnn = {'batch_size': 8, 'learning_rate': 0.001, 'optimizer': 'Adam', 'dropout': 0.2, 'weight_decay': 0.001}
best_conf_convnext= {'batch_size': 8, 'learning_rate': 0.001, 'optimizer': 'Adam', 'dropout': 0.2, 'weight_decay': 0.001}
best_conf_vit = {'batch_size': 8, 'learning_rate': 0.001, 'optimizer': 'Adam', 'dropout': 0.2, 'weight_decay': 0.001}

best_confs = {
    "ResNet": best_conf_resnet,
    "ConvNeXt": best_conf_convnext,
    "VisionTransformer": best_conf_vit,
    "SmallCNN": best_conf_smallcnn
}

In [15]:
# standard operations
basic_augmentations = ["flip", "shift", "rotation"]
# more advanced data augmentation
advanced_augmentations = 'cutmix' 

In [ ]:
def run_experiment_augmentation(model_name,experiment_config,augmentation=None, cutmix_collate_fn=None, epoch_number=2):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # model
    model = get_model(model_name)   
    model.to(device)


    # optimizer
    if experiment_config["optimizer"] == "adam":
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"]
        )
    else:
        optimizer = torch.optim.SGD(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"],
            momentum=0.9
        )

    # data
    transformations = basic_transforms(augmentation_type=augmentation, model_type=model_name)


    train_dataset = get_train_dataset(transform=transformations)
    val_dataset, test_dataset = get_val_train_dataset(model_type=model_name)

    

    train_loader = get_train_dataloaders(
        train_dataset,
        collate_fn=cutmix_collate_fn,
        batch_size=experiment_config["batch_size"]
    )
    
    val_loader, test_loader = get_val_test_dataloaders(
        val_dataset, test_dataset,
        batch_size=experiment_config["batch_size"]
    )

    criterion = torch.nn.CrossEntropyLoss()

    accs = []
    f1s = []
    recalls = []
    precisions = []
    for epoch in range(epoch_number):

        loss_score = train(model, train_loader, optimizer, criterion, device)
        scores = validate(model, val_loader, device, criterion)

        print(f"Epoch number: {epoch}; training loss: {loss_score:.5f}; val loss: {scores['loss']:.5f}")


        accs.append(scores["accuracy"])
        f1s.append(scores["f1"])
        recalls.append(scores["recall"])
        precisions.append(scores["precision"])

    val_scores = {
        'mean_acc': mean(accs),
        'std_acc': stdev(accs),

        'mean_f1': mean(f1s),
        'std_f1': stdev(f1s),

        'mean_recall': mean(recalls),
        'std_recall': stdev(recalls),
        'mean_precision': mean(precisions),
        'std_precision': stdev(precisions)
    }

    test_scores = validate(model, test_loader, device, criterion)

    return {**val_scores, **test_scores}

In [ ]:
# augmentation experiments

results_augm = []
for model in models:
    for augm in basic_augmentations:

        print(f"Model: {model}; Augmentation: {augm}")


        score = run_experiment_augmentation(model,best_confs[model], augm, cutmix_collate_fn=None)
        d = {
            "model": model,
            "augmentation": augm}
        
        results_augm.append({
            **d, **score
        })

    print(f"Model: {model} augmentation: cutmix")
    score = run_experiment_augmentation(model,best_confs[model], None, cutmix_collate_fn=cutmix_collate_fn)
    d = {
        "model": model,
        "augmentation": 'cutmix'}
        
    results_augm.append({
            **d, **score
        })

## Model Ranking

In [ ]:
for model in models:
    model_results = list(filter(lambda res: res["model"] == model, results_augm))
    sorted_results = sorted(model_results, key=lambda x: -x["accuracy"])
    print(f"Ranking for model {model}")
    for i,res in enumerate(sorted_results):
        print("Rank number: ",i, end=" ")
        print(sorted_results[i])
    print("="*50)

Ranking for model ResNet
Rank number:  0 {'model': 'ResNet', 'augmentation': 'shift', 'mean_acc': 0.2833333333333333, 'std_acc': 0.07071067811865474, 'mean_f1': 0.19761650886330612, 'std_f1': 0.04490380308558294, 'mean_recall': 0.2555672268907563, 'std_recall': 0.036068387914305354, 'mean_precision': 0.2562156862745098, 'std_precision': 0.12687714028702146, 'loss': 2.2505622704823813, 'accuracy': 0.25555555555555554, 'precision': 0.2663059163059163, 'recall': 0.31682539682539684, 'f1': 0.21395224017175235}
Rank number:  1 {'model': 'ResNet', 'augmentation': 'rotation', 'mean_acc': 0.11666666666666667, 'std_acc': 0.07071067811865475, 'mean_f1': 0.06727716727716729, 'std_f1': 0.0762881015697721, 'mean_recall': 0.15531135531135531, 'std_recall': 0.07822206883455578, 'mean_precision': 0.09055829228243022, 'std_precision': 0.11796723968563749, 'loss': 2.2699051002661386, 'accuracy': 0.2222222222222222, 'precision': 0.13600454674623472, 'recall': 0.19007936507936507, 'f1': 0.1317955033472274

In [ ]:
# tutaj też sprawdzamy jak się modele zachowywały, wizualizacje i tym podobne etc.

## Few-shot learning

In [ ]:
#przyklad jak wybrać podzbior do fewshot learningu, wystarczy na wczytanym datasecie zastosować few_shot_subset

transformations = basic_transforms(augmentation_type=None, model_type='ResNet')

train_dataset = get_train_dataset(transform=transformations)
val_dataset, test_dataset = get_val_train_dataset(model_type='ResNet')

few_shot_train =  few_shot_subset(train_dataset)
few_shot_val =  few_shot_subset(val_dataset)
few_shot_test =  few_shot_subset(test_dataset)